# Data Check: NSW Crash Data 2024 Onward

This notebook reads `data/nsw_crash_data_clean.csv`, standardises the column names, filters the dataset to **2024 onward** and saves a filtered version for the Streamlit app.

Requirement applied: keep records where `year_of_crash >= 2024`. This means 2024 is included, while 2023 and earlier are removed.

In [ ]:
# 1. Load packages
import os
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

In [ ]:
# 2. Define paths
CSV_PATH = "data/nsw_crash_data_clean.csv"
OUTPUT_PATH = "data/nsw_crash_data_2024_onward.csv"

print("Input file exists:", os.path.exists(CSV_PATH))
print("Input path:", CSV_PATH)

In [ ]:
# 3. Load dataset
df = pd.read_csv(CSV_PATH, low_memory=False)

print("Original shape:", df.shape)
display(df.head())

In [ ]:
# 4. Standardise column names
# This keeps the file easier to use in Python, Streamlit and Tableau.
df.columns = (
    df.columns.astype(str)
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

print("Number of columns:", len(df.columns))
print(df.columns.tolist())

In [ ]:
# 5. Check required columns
required_cols = [
    "crash_id",
    "year_of_crash",
    "month_of_crash",
    "day_of_week_of_crash",
    "two-hour_intervals",
    "degree_of_crash",
    "lga",
    "latitude",
    "longitude",
    "school_zone_location",
    "school_zone_active",
    "weather",
    "natural_lighting",
    "surface_condition",
    "speed_limit",
]

missing_required = [col for col in required_cols if col not in df.columns]

if missing_required:
    print("Missing required columns:", missing_required)
else:
    print("All required columns are available.")

In [ ]:
# 6. Clean key numeric fields
numeric_cols = [
    "year_of_crash",
    "latitude",
    "longitude",
    "no._killed",
    "no._seriously_injured",
    "no._moderately_injured",
    "no._minor-other_injured",
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print(df[numeric_cols[:3]].dtypes)

In [ ]:
# 7. Filter to 2024 onward
# Keep 2024 and later only. This removes 2023 and earlier.
df_2024 = df[df["year_of_crash"] >= 2024].copy()

print("Original shape:", df.shape)
print("2024 onward shape:", df_2024.shape)
print("Years included:", sorted(df_2024["year_of_crash"].dropna().unique().tolist()))

In [ ]:
# 8. Add month number and Australian season fields
month_order = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December"
]
month_to_num = {month: i + 1 for i, month in enumerate(month_order)}

def get_au_season(month_number):
    if pd.isna(month_number):
        return "Unknown"
    month_number = int(month_number)
    if month_number in [12, 1, 2]:
        return "Summer"
    if month_number in [3, 4, 5]:
        return "Autumn"
    if month_number in [6, 7, 8]:
        return "Winter"
    if month_number in [9, 10, 11]:
        return "Spring"
    return "Unknown"

if "month_of_crash" in df_2024.columns:
    df_2024["month_num"] = df_2024["month_of_crash"].map(month_to_num)
    df_2024["season"] = df_2024["month_num"].apply(get_au_season)

display(df_2024[["year_of_crash", "month_of_crash", "month_num", "season"]].head())

In [ ]:
# 9. Data quality checks
print("Duplicate crash_id:", df_2024["crash_id"].duplicated().sum() if "crash_id" in df_2024.columns else "crash_id not found")
print("Missing latitude:", df_2024["latitude"].isna().sum() if "latitude" in df_2024.columns else "latitude not found")
print("Missing longitude:", df_2024["longitude"].isna().sum() if "longitude" in df_2024.columns else "longitude not found")

missing_summary = (
    df_2024.isna().sum()
    .sort_values(ascending=False)
    .reset_index()
)
missing_summary.columns = ["column", "missing_count"]
missing_summary["missing_%"] = (missing_summary["missing_count"] / len(df_2024) * 100).round(2)

display(missing_summary.head(20))

In [ ]:
# 10. Key summaries for dashboard planning
summary_tables = {}

for col in [
    "degree_of_crash",
    "month_of_crash",
    "season",
    "day_of_week_of_crash",
    "two-hour_intervals",
    "lga",
    "school_zone_location",
    "school_zone_active",
    "weather",
    "natural_lighting",
    "surface_condition",
    "speed_limit",
]:
    if col in df_2024.columns:
        summary_tables[col] = df_2024[col].fillna("Unknown").value_counts().reset_index()
        summary_tables[col].columns = [col, "crash_count"]
        print("\n---", col, "---")
        display(summary_tables[col].head(15))

In [ ]:
# 11. Save filtered data for Streamlit/Tableau
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
df_2024.to_csv(OUTPUT_PATH, index=False)

print("Saved filtered file to:", OUTPUT_PATH)
print("Saved shape:", df_2024.shape)

In [ ]:
# 12. Final preview
display(df_2024.head())